# cv-data-lesson 授業用ランタイム受け入れテスト

授業で使う通常の導入方法 `%pip install cv-data-lesson` から始め、公開版のインストール結果と、これまでの指摘事項をまとめて確認するNotebookです。インストール後のバージョンは固定せず表示だけします。

## 実行するランタイム

1. まずColabの **ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: なし** を選び、CPUでデータ・bbox・ResNet・Tiny-SDを実行します。
2. 次に **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選び、接続し直して、このNotebookを最初のセルからもう一度実行します。T4へ切り替えた後に途中のセルだけを続けず、パッケージとモデルのdevice判定をやり直してください。
3. OCRはPaddlePaddleとPyTorch/Diffusersを同じランタイムで保証しないため、最後のOCR専用セクションを **新しいCPUランタイム** で実行します。

ランタイムを切り替えると `/content` のインストール・モデルキャッシュは失われます。毎回最初から実行してください。データやモデルをGitリポジトリへ保存する操作はありません。

In [ ]:
import sys
print("Python:", sys.version)
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Python 3.11〜3.13のランタイムを選択してください"

## 1. 授業と同じ通常インストール

データカタログと固定Releaseの取得だけなら、下の通常インストールで動きます。DataFrameやモデル機能は仕様どおり追加依存を分けています。

In [ ]:
%pip install -q --upgrade cv-data-lesson

In [ ]:
%pip install -q --upgrade "cv-data-lesson[dataframe,display]"

In [ ]:
import importlib.metadata as metadata
from pathlib import Path

import cvlesson
from cvlesson import datasets

INSTALLED_VERSION = metadata.version("cv-data-lesson")
print("cv-data-lesson:", INSTALLED_VERSION)
print("module:", cvlesson.__file__)
assert cvlesson.__file__ is not None
print("通常pip導入: OK")

## 2. カタログ・日本語ラベル・データ不整合の確認

カタログ表示ではダウンロードしません。ラベル表は元ラベルを残したまま日本語名が全クラスにあるか確認します。特に、FruitSeg30・Indoor Waste・Fresh and Rotten Fruits・Orange・Fire・AgriFreshNETの指摘ラベルを確認します。

In [ ]:
catalog = datasets.list_available()
catalog_ids = {item["id"] for item in catalog}
required_ids = {"geoset_ennea", "rdd2020_japan", "fruitseg30", "indoor_waste_image", "fresh_rotten_fruits"}
assert required_ids <= catalog_ids

missing_labels = []
for item in catalog:
    source = datasets._source(item["id"])
    labels_ja = dict(zip(source.classes, source.class_names_ja))
    labels_ja.update(source.label_names_ja or {})
    missing_labels.extend((item["id"], label) for label in source.classes if not labels_ja.get(label))
assert not missing_labels, missing_labels

expected_labels = {
    ("orange_fruit", "cankered_orange"): "傷んだオレンジ",
    ("orange_fruit", "fresh_orange"): "新鮮なオレンジ",
    ("orange_fruit", "rotten_orange"): "腐ったオレンジ",
    ("fruitseg30", "Apple_Gala"): "りんご（ガラ種）",
    ("fruitseg30", "Apple_Golden Delicious"): "りんご（ゴールデンデリシャス種）",
    ("fruitseg30", "Green Coconut"): "緑色のココナッツ",
    ("fruitseg30", "Malta"): "マルタオレンジ",
    ("indoor_waste_image", "shell"): "ナッツの殻",
    ("fresh_rotten_fruits", "fresh_apple"): "新鮮なリンゴ",
    ("fresh_rotten_fruits", "rotten_apple"): "傷んだリンゴ",
    ("fire_recognition_image", "Artificial_Fire"): "擬似的な炎",
    ("fire_recognition_image", "Fire"): "炎",
    ("fire_recognition_image", "Safe_Fire"): "日常で扱われる火",
    ("agrifreshnet", "Semi fresh Orange(9-20)"): "少し傷んだオレンジ",
}
for (dataset_id, raw_label), expected_ja in expected_labels.items():
    source = datasets._source(dataset_id)
    labels_ja = dict(zip(source.classes, source.class_names_ja))
    labels_ja.update(source.label_names_ja or {})
    assert labels_ja[raw_label] == expected_ja, (dataset_id, raw_label, labels_ja[raw_label])
print("catalog entries:", len(catalog))
print("all public classes have Japanese labels: OK")
print("selected raw/Japanese labels: OK")

## 3. GeoSetEnnea / RDD2020の取得、split、bbox、ファイル整合性

固定Releaseの全量を取得します。GeoSetEnneaは `val`、RDD2020は `valid` の元表記を持つため、両方を受け付け、ライブラリの `split` と集計は `val` に統一し、元表記は `source_split` に残すことを確認します。bboxは小さな丸め誤差だけを許容し、NaN・無限大・大きな範囲外値・不明クラス・欠落ファイルを検出します。

In [ ]:
DATA_CACHE = Path("/content/cvlesson-lesson-data-cache")
geo = datasets.get("geoset_ennea", cache_dir=DATA_CACHE)
rdd = datasets.get("rdd2020_japan", cache_dir=DATA_CACHE)
geo_df = geo.to_df()
rdd_df = rdd.to_df()

assert geo.splits == {"train": 84, "val": 22, "test": 30}
assert rdd.splits == {"train": 6299, "val": 802, "test": 799}
assert set(geo_df["split"].unique()) == {"train", "val", "test"}
assert set(rdd_df["split"].unique()) == {"train", "val", "test"}
assert set(geo_df["source_split"].unique()) == {"train", "val", "test"}
assert set(rdd_df["source_split"].unique()) == {"train", "valid", "test"}
print("GeoSetEnnea normalized/source:", geo.splits, set(geo_df["source_split"].unique()))
print("RDD2020 normalized/source:", rdd.splits, set(rdd_df["source_split"].unique()))
print("val normalization and source split retention: OK")

In [ ]:
def check_detection_data(bundle, frame, expected_rows, expected_boxes):
    assert len(frame) == expected_rows
    assert {"image_path", "label_file", "boxes", "split", "sample_id"} <= set(frame.columns)
    assert "class_name" not in frame.columns, "class_nameはトップレベルではなくboxes内です"
    assert frame["sample_id"].is_unique
    names_ja = dict(zip(bundle.classes, bundle.class_names_ja))
    box_count = 0
    for row in frame.itertuples(index=False):
        image_path = Path(row.image_path)
        label_file = Path(row.label_file)
        assert image_path.is_file(), image_path
        assert label_file.is_file(), label_file
        assert row.split in {"train", "val", "test"}
        for box in row.boxes:
            box_count += 1
            assert 0 <= int(box["class_id"]) < len(bundle.classes)
            class_id = int(box["class_id"])
            assert box["class_name"] == bundle.classes[class_id]
            assert box["class_name_ja"] == names_ja[bundle.classes[class_id]]
            values = [float(box[key]) for key in ("x_center", "y_center", "width", "height")]
            assert all(value == value and abs(value) != float("inf") for value in values)
            x_center, y_center, width, height = values
            assert width > 0 and height > 0
            assert -1e-9 <= x_center - width / 2 <= 1 + 1e-9
            assert -1e-9 <= y_center - height / 2 <= 1 + 1e-9
            assert -1e-9 <= x_center + width / 2 <= 1 + 1e-9
            assert -1e-9 <= y_center + height / 2 <= 1 + 1e-9
    assert box_count == expected_boxes
    return box_count

geo_boxes = check_detection_data(geo, geo_df, 136, 1188)
rdd_boxes = check_detection_data(rdd, rdd_df, 7900, 16469)
offline_rdd = datasets.get("rdd2020_japan", cache_dir=DATA_CACHE, allow_download=False)
assert offline_rdd.root == rdd.root
print("GeoSetEnnea bbox/files: OK (", geo_boxes, ")")
print("RDD2020 bbox/files/range: OK (", rdd_boxes, ")")
print("data inconsistency checks: OK")

## 4. モデル用追加依存を通常版から導入

仕様上、PyTorch・DETR/SAM・Diffusersは基本パッケージへ自動同梱しません。授業でモデルを使う場合だけ、バージョン番号を付けずに追加します。OCRはPaddle専用ランタイムで別に行います。

In [ ]:
%pip install -q --upgrade "cv-data-lesson[dataframe,display,torch,vision,diffusion]"

In [ ]:
import gc
import importlib.util
import warnings

import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from cvlesson import generation, models, vision

HAS_GPU = bool(torch.cuda.is_available())
MODEL_CACHE = Path("/content/cvlesson-lesson-model-cache")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
print("runtime:", "T4/GPU" if HAS_GPU else "CPU")
print("ResNet devices:", models.available_devices())
print("Vision devices:", vision.available_devices())
print("Generation devices:", generation.available_devices())
assert models.available_devices()["cpu"]
assert vision.available_devices()["cpu"]
assert generation.available_devices()["cpu"]
assert importlib.util.find_spec("timm") is not None, "vision追加機能にtimmがありません"
if HAS_GPU:
    assert models.available_devices()["cuda"]
    assert vision.available_devices()["cuda"]
    assert generation.available_devices()["cuda"]
print("optional dependencies, timm, and runtime devices: OK")

## 5. ResNetのdevice表記・進捗・分類/回帰

PyTorch標準の`cuda`を第1GPUの公開名にします。入力の`cuda:0`も同じGPUの別名として受け付けますが、利用可能デバイス一覧では重複表示しません。実際に小さなデータで分類と画像ペア回帰を1エポックだけ行い、進捗表示と予測経路を確認します。

In [ ]:
assert models.ResNetClassifier(device="cuda", weights=None).device == "cuda"
classification_images = [Image.new("RGB", (96, 96), color) for color in ("red", "red", "blue", "blue")]
classification_labels = ["赤", "赤", "青", "青"]
classifier = models.ResNetClassifier(device="auto", epochs=1, batch_size=2, weights=None, verbose=True)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    classifier.fit(classification_images, classification_labels)
assert classifier.device == ("cuda" if HAS_GPU else "cpu")
if not HAS_GPU:
    assert any("CPU" in str(item.message) for item in captured)
assert len(classifier.predict(classification_images)) == 4
assert classifier.predict_proba(classification_images).shape == (4, 2)
print("ResNet classification device:", classifier.device)
print("ResNet classification/progress: OK")

regression_images = [(Image.new("RGB", (96, 96), "black"), Image.new("RGB", (96, 96), "white")), (Image.new("RGB", (96, 96), "white"), Image.new("RGB", (96, 96), "black"))]
regressor = models.ResNetRegressor(device="auto", epochs=1, batch_size=1, weights=None, verbose=True)
regressor.fit(regression_images, [0.25, 0.75])
assert regressor.predict(regression_images).shape == (2,)
print("ResNet regression: OK")
del classifier, regressor
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

## 6. DETR/SAMのCPU・T4自動切替

同じ固定revisionモデルを、T4があればGPU、なければ警告を出してCPUへ切り替えます。CPUでは時間がかかります。

In [ ]:
vision_image = Image.new("RGB", (256, 256), "white")
vision_draw = ImageDraw.Draw(vision_image)
vision_draw.rectangle((48, 48, 208, 208), fill="red")
detector = vision.load_detector(device="auto", cache_dir=MODEL_CACHE)
assert detector.device == ("cuda" if HAS_GPU else "cpu")
detections = detector.predict(vision_image, threshold=0.0)
assert isinstance(detections, list)
print("DETR device:", detector.device, "detections:", len(detections))
segmenter = vision.load_segmenter(device="auto", cache_dir=MODEL_CACHE)
assert segmenter.device == ("cuda" if HAS_GPU else "cpu")
segmentation = segmenter.predict(vision_image, box=(48, 48, 208, 208))
assert segmentation.mask.shape == (256, 256)
assert segmentation.mask.dtype == bool
print("SAM device:", segmenter.device, "mask shape:", segmentation.mask.shape)
print("DETR/SAM auto fallback: OK")
del detector, segmenter
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

## 7. 画像生成のGPU/CPU境界

`model="auto"`はT4でSDXL、CPUでLoRAなしのTiny-SDになります。共通のprompt・seed・steps・guidance・画像サイズは使えます。Tiny-SDでLoRAやimage-to-imageを指定した場合と、CPUでSDXLを明示した場合は、黙って切り替えずエラーにします。

In [ ]:
with warnings.catch_warnings(record=True) as generation_warnings:
    warnings.simplefilter("always")
    generator = generation.load(model="auto", device="auto", lora=None, cache_dir=MODEL_CACHE)
assert generator.model == ("sdxl" if HAS_GPU else "tiny-sd")
assert generator.device == ("cuda" if HAS_GPU else "cpu")
if not HAS_GPU:
    assert any("Tiny-SD" in str(item.message) for item in generation_warnings)
generated = generator.generate("a red apple on a white table", seed=42, num_inference_steps=2, guidance_scale=5.0, width=256, height=256)
assert isinstance(generated, Image.Image) and generated.size == (256, 256)
display(generated)
print("generation model/device:", generator.model, generator.device)
print("image generation auto fallback: OK")

In [ ]:
try:
    generation.load(model="sdxl", device="cpu")
except generation.GenerationError as error:
    print("explicit SDXL on CPU rejection: OK -", error)
else:
    raise AssertionError("SDXLのCPU明示指定が拒否されませんでした")
try:
    generation.load(model="tiny-sd", device="cpu", lora="fluent_emoji_3d")
except ValueError as error:
    print("Tiny-SD LoRA rejection: OK -", error)
else:
    raise AssertionError("Tiny-SDがLoRA指定を受け入れました")
try:
    generation.load(model="tiny-sd", device="cpu", task="image-to-image", local_files_only=True)
except ValueError as error:
    print("Tiny-SD image-to-image rejection: OK -", error)
else:
    raise AssertionError("Tiny-SDがimage-to-image指定を受け入れました")

In [ ]:
if not HAS_GPU:
    try:
        models.ResNetClassifier(device="cuda", epochs=1, weights=None).fit(
            [Image.new("RGB", (64, 64), "red"), Image.new("RGB", (64, 64), "blue")], ["赤", "青"]
        )
    except models.ModelError as error:
        print("explicit ResNet GPU rejection: OK -", error)
    else:
        raise AssertionError("利用できないGPU指定がCPUへ黙って切り替わりました")
    try:
        vision.load_detector(device="cuda", local_files_only=True)
    except vision.VisionError as error:
        print("explicit DETR GPU rejection: OK -", error)
    else:
        raise AssertionError("利用できないDETR GPU指定が受け入れられました")
else:
    print("T4ではcuda/cuda:0が利用可能であることを環境セルで確認済み")

## 8. OCR専用ランタイム

ここからはメインのランタイムを切断し、新しいCPUランタイムで実行します。PaddlePaddleはCPU/GPUやCUDAに合う公式パッケージが必要です。下はCPU版の例です。通常導入と同じく、cv-data-lessonのバージョン指定はありません。

In [ ]:
%pip install -q "paddlepaddle==3.2.0" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip install -q --upgrade "cv-data-lesson[ocr,dataframe,display]"

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from cvlesson import ocr

OCR_CACHE = Path("/content/cvlesson-lesson-ocr-cache")
reader = ocr.load(cache_dir=OCR_CACHE)
assert reader.device == "cpu"
ocr_image = Image.new("RGB", (1000, 240), "white")
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = ImageFont.truetype(font_path, 92) if Path(font_path).exists() else ImageFont.load_default()
ImageDraw.Draw(ocr_image).text((40, 55), "TOTAL 1280 JPY", fill="black", font=font)
ocr_dir = Path("/content/cvlesson-lesson-ocr-inputs")
ocr_dir.mkdir(parents=True, exist_ok=True)
webp_path = ocr_dir / "sample.webp"
ocr_image.save(webp_path, format="WEBP")
result = reader.predict(webp_path)
assert result.text.strip(), "WebPが空結果になっています"
broken_path = ocr_dir / "broken.webp"
broken_path.write_bytes(b"not an image")
try:
    reader.predict(broken_path)
except ocr.OCRError as error:
    print("OCR WebP and broken-image validation: OK -", error)
else:
    raise AssertionError("壊れた画像が空のOCR結果として処理されました")

In [ ]:
# 共通表示APIの実動作を確認する。
import tempfile
import numpy as np
from PIL import Image
from cvlesson import display as cv_display
from cvlesson.ocr import OCRLine, OCRResult
from cvlesson.vision import Detection, Segmentation
display_fixture = Image.new("RGB", (128, 128), "white")
display_detection = Detection("fixture", 0.9, (16, 20, 96, 110))
display_mask = np.zeros((128, 128), dtype=bool)
display_mask[32:96, 32:96] = True
display_segmentation = Segmentation(display_mask, 0.8, "box", display_detection.box)
display_ocr = OCRResult((OCRLine("確認", 0.95, ((8, 8), (70, 8), (70, 28), (8, 28))),), "fixture")
rendered = cv_display.render(display_fixture, [display_detection, display_segmentation, display_ocr])
assert rendered.size == display_fixture.size
with tempfile.TemporaryDirectory() as temporary_directory:
    saved_path = cv_display.save(display_fixture, Path(temporary_directory) / "overlay.png", [display_detection, display_segmentation, display_ocr])
    assert saved_path.is_file()
shown = cv_display.show(display_fixture, display_detection)
assert shown.size == display_fixture.size
print("display render/show/save: OK")

## 判定

CPU/T4のメインセクションと、OCR専用ランタイムのセクションが成功すれば、授業で使う通常pip導入に対して、指摘されていたデータ不整合・bbox・ラベル・device・CPU/GPU・Tiny-SD・OCR入力の確認ができます。これは実行した環境と入力に対する受け入れテストであり、全データ・全画像・全プロンプトの精度を保証するものではありません。失敗時は、失敗セル全文、`runtime`表示、`cv-data-lesson`表示バージョンを記録してください。